# ESS Battery Health 

데이터셋 : Stanford & MIT - *Data-driven prediction of battery cycle life before capacity degradation* (Nature Energy, 2019)  

> 재사용 가능한 계산은 `src/skala_ds_mini_project/`에 있습니다. 저장된 출력은 리팩터링 후 모두 제거했습니다. 프로젝트 루트에서 `uv sync` 후 같은 환경의 Jupyter 커널로 위에서부터 실행하세요. EDA 플롯은 노트북에 유지하며, 대용량 MAT 파일은 feature 필드만 배치별로 읽습니다.


In [ ]:
# 프로젝트 루트에서 uv sync 후 같은 환경의 Jupyter 커널을 사용합니다.

In [ ]:
import os
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from skala_ds_mini_project.analysis import find_knee_point
from skala_ds_mini_project.data import (
    BATCH_FILES,
    extract_summary,
    load_feature_batch,
    normalize_batch,
    to_list_of_dicts,
)
from skala_ds_mini_project.features import (
    FEATURE_COLUMNS,
    avg_c_rate,
    build_feature_table,
    fade_slope,
    load_feature_inputs,
    parse_policy,
)
from skala_ds_mini_project.preprocessing import estimate_nominal_capacity, filter_capacity

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 12

In [ ]:
# 데이터 경로 설정
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"  # 외부 데이터 경로를 사용할 경우 이 값만 변경

files = sorted(os.listdir(DATA_DIR))
print(f"데이터 경로 : {DATA_DIR}")
print("파일 목록 :")
for f in files:
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024**3)
    print(f"   {f}  ({size_mb:.1f} GB)")

## Loading


- `.mat` 파일은 MATLAB 형식이며, 이 파일은 **MATLAB v7.3(HDF5)** 포맷임
- 먼저 `scipy.io.loadmat()`을 시도하고, v7.3이면 `mat73`로 fallback 함
- 파일 크기가 크므로 **Batch 1** 하나만 먼저 로드함

In [ ]:
# Batch 1: summary와 Qdlin cycle 10/100만 로드 (raw I/V/t 파형 제외)
batch1_path = DATA_DIR / BATCH_FILES[0]
batch = load_feature_batch(batch1_path)
print(f"로드된 배터리 셀 수 : {len(batch)}")

## 데이터 타입 및 구조 확인

이 데이터셋은 `batch` 키 아래 배터리 셀 배열로 구성되어 있음

In [ ]:
batch = normalize_batch(batch)
print(f"배터리 셀 수    : {len(batch)}")
print(f"타입            : {type(batch)}")
print(f"첫 번째 셀 키   : {list(batch[0].keys())}")

In [ ]:
# 단일 셀 구조 상세 확인
cell0 = batch[0]

print("=" * 55)
print("배터리 셀[0] 구조")
print("=" * 55)

for key, val in cell0.items():
    if isinstance(val, np.ndarray):
        print(f"  {key:25s} | ndarray  | shape={val.shape}")
    elif isinstance(val, dict):
        print(f"  {key:25s} | dict     | keys={list(val.keys())}")
    else:
        print(f"  {key:25s} | {type(val).__name__:8s} | {val}")

In [ ]:
# 선택 로드된 Qdlin 사이클 구조 확인 (미선택 사이클 위치는 None)
cycles_raw = cell0["cycles"]
cycles = to_list_of_dicts(cycles_raw)
print(f"전체 summary 사이클 수 : {len(np.asarray(cell0['summary']['QDischarge']).reshape(-1))}")
print("\n선택 로드된 cycle 10 내 변수 목록 :")
for key, val in cycles[9].items():
    if isinstance(val, np.ndarray):
        print(f"  {key:15s} | ndarray | shape={val.shape} | dtype={val.dtype}")
    else:
        print(f"  {key:15s} | {type(val).__name__:8s} | {val}")

### Features

- Descriptors : 배터리 수준 스칼라
    - `cycle_life` : EOL(80% 용량)까지 총 사이클 수 ⇒ Target Variable
    - `charging_policy` : 충전 프로토콜 (e.g. : `"4.8C(80%)-3.6C"`- 용량의 80%까지 4.8C 속도로 빠르게 충전하고, 이후 나머지는 3.6C로 천천히 충전한다)
- Summary : 사이클별 요약 스칼라
    - `QD` : 방전 용량 (Ah)
    - `Qc` : 충전 용량 (Ah)
    - `IR` : 내부 저항 (Ohm)
    - `Tmax` / `Tavg` / `Tmin` : 사이클 내 최고, 평균, 최저 온도
    - `chargetime` : 충전 소요 시간
    - `discharge_time` : 방전 소요 시간
- Cycles : 사이클 내부 시계열
    - `t` : 시간 (분)
    - `V` : 전압 (V)
    - `I` : 전류 (A)
    - `T` : 온도 (°C)
    - `Qc`/ `Qd` : 충전/방전 용량
    - `Qdlin` : `ΔQ(V)` 곡선 계산의 기반 데이터, 전압 축으로 선형 보간된 방전 용량 (1,000포인트, 2V ~ 3.6V)
    - `Tdlin` : 전압 축으로 선형 보간된 온도 (1,000 포인트)
    - `discharge_dQdV` : 방전 dQ/dV 곡선 (파생변수)

## Summary 데이터 추출

각 배터리의 `summary` 필드에서 사이클별 요약 지표를 DataFrame으로 변환

In [ ]:
# summary 추출 구현은 skala_ds_mini_project.data.extract_summary에서 재사용합니다.

In [ ]:
df = extract_summary(batch)

print(f"DataFrame shape : {df.shape}")
df.head(n=10)

In [ ]:
# 기본 통계
df.describe().round(3)

### MEMO

#### QC, QD

    - `QD` : 방전 용량 (Ah)
    - `QD` : 충전 용량 (Ah)

> 평균에 비해 MAX가 너무 크다.

#### IR

    - `IR` : 내부 저항 (Ohm)

> mean이 0인게 있다 (#정인) -> 결측치인가? 측정오류인가? 0 개수 확인 해야할 듯

#### Temperature

    - `Tmax` / `Tavg` / `Tmin` : 사이클 내 최고, 평균, 최저 온도

> Tmax, Tavg -> 조금 큰 놈들이 있다.

#### Chargetime

    - `chargetime` : 충전 소요 시간

> 극단적인 outlier가 존재

In [ ]:
print("결측치 현황 : ")
print(df.isnull().sum())

In [ ]:
print(f"\n배터리 셀 수   : {df['cell_id'].nunique()}")
print(f"충전 정책 종류 : {df['charging_policy'].nunique()}")

## EDA(Basic)

### 1. Cycle Life 분포

In [ ]:
cycle_life_df = df.drop_duplicates("cell_id")[["cell_id", "cycle_life", "charging_policy"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
axes[0].hist(cycle_life_df["cycle_life"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Histogram")
axes[0].set_xlabel("Cycle Life")
axes[0].set_ylabel("num of Battery Cell")
axes[0].axvline(
    cycle_life_df["cycle_life"].mean(),
    color="tomato",
    linestyle="--",
    label=f"mean = {cycle_life_df['cycle_life'].mean():.0f}",
)
axes[0].legend()

# Box Plot
axes[1].boxplot(
    cycle_life_df["cycle_life"], patch_artist=True, boxprops=dict(facecolor="steelblue", alpha=0.6)
)
axes[1].set_title("Box Plot")
axes[1].set_ylabel("Cycle Life")
axes[1].set_xticks([])

plt.suptitle("Cycle Life", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(cycle_life_df["cycle_life"].describe().round(1))

[describe] 
- 배터리 평균 수명 약 845 사이클 (단명 534 ~ 장수 1227)
- 평균 vs 중앙값 차이 없음 (데이터 왜곡/편향 적은 편) 
- IQR 통해 절반의 배터리가 703 ~ 914 범위에 집중 

[plot]
- Histogram : 845 근처에 가장 몰려 있고 600대와 1,200대에 일부 분포. 완전한 정규분포는 아님 
- Box plot : 
    - Upper whisker = |75%(914) - max(1227)| = 313
    - Lower whisker = |25%(703) - min(534) | = 169
- Upper가 큼 -> 장수 배터리의 수명 편차가 더 큼 -> 좋은 충전 조건을 만나면 수명이 크게 늘어날 수 있지만, 그 폭이 불규칙함  
- Lower는 상대적으로 짧음 -> 단명 배터리가 534~703 구간에 비교적 일정하게 분포 -> 나쁜 충전 조건에서는 수명이 어느 정도 예측 가능한 범위로 떨어질 수 있음 

[to modeling]
- 수명이 긴 배터리일수록 예측 난이도 있음 
- residual 분석을 함께 진행하는 것도 필요할 수 있음 

### 2. 열화 곡선 - 방전 용량(QD) 감소 패턴

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

cell_ids = df["cell_id"].unique()
colors = cm.coolwarm(np.linspace(0, 1, len(cell_ids)))

for i, cid in enumerate(cell_ids):
    sub = df[df["cell_id"] == cid]
    ax.plot(sub["cycle"], sub["QD"], color=colors[i], linewidth=0.8, alpha=0.7)

nominal = df[df["cycle"] <= 5].groupby("cell_id")["QD"].mean().mean()
ax.axhline(
    y=0.88 * nominal,  # ~80% of nominal ≈ 1.1Ah
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="EOL(80%)",
)
ax.set_xlabel("Cycle")
ax.set_ylabel("QD (Ah)")
ax.set_title("Battery Cell - QD Curve")
ax.legend()
plt.tight_layout()
plt.show()

[plot]
- 초기 이상값 스파이크 (cycle 0 ~ 50 구간) : 정상적인 배터리 용량을 벗어나므로, outlier filtering 고려 대상 
- 열화 패턴 : 대부분의 셀이 약 1.1Ah에서 시작해 사이클이 늘어날수록 원만하게 감소함. 논문에서 언급된 용량 열화(Capacity Fade) 곡선 형태 확인됨 
- 색상 분포 : 파란색은 장수 배터리, 빨간색은 단명 배터리로 구분됨. 사이클 600 이후에 파란색이 높게 유지되는 것으로 보임 

In [ ]:
# Outlier 필터링 후 열화 곡선

# 1. 정상 용량 범위 정의
nominal = estimate_nominal_capacity(df)
lower = nominal * 0.80
upper = nominal * 1.20

print(f"공칭 용량(nominal) : {nominal:.4f} Ah")
print(f"필터 범위          : {lower:.4f} ~ {upper:.4f} Ah")

# 2. 필터링
df_clean = filter_capacity(df, lower=0.80, upper=1.20, nominal=nominal)
print(
    f"제거된 행 수       : {len(df) - len(df_clean)} ({(len(df) - len(df_clean)) / len(df) * 100:.2f}%)"
)

# 3. 시각화
fig, ax = plt.subplots(figsize=(14, 6))

cell_ids = df_clean["cell_id"].unique()
colors = cm.coolwarm(np.linspace(0, 1, len(cell_ids)))

for i, cid in enumerate(cell_ids):
    sub = df_clean[df_clean["cell_id"] == cid]
    ax.plot(sub["cycle"], sub["QD"], color=colors[i], linewidth=0.8, alpha=0.7)

ax.axhline(
    y=nominal * 0.8,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"EOL({nominal * 0.8:.2f} Ah)",
)
ax.set_xlabel("Cycle")
ax.set_ylabel("QD (Ah)")
ax.set_title("Battery Cell - QD Curve (Outlier filtering)")
ax.set_ylim(0.7, 1.2)
ax.legend()
plt.tight_layout()
plt.show()

[plot]
- cycle  1 ~  50 : 모든 셀이 1.06~1.10Ah 범위에서 시작하며 초기 용량이 매우 균일함 -> 셀 간 초기 상태 차이 거의 없음 
- cycle 50 ~ 400 : 회색 계열 셀들이 다른 셀보다 빠르게 하강
- cycle 400 ~    : 
    - 붉은 계열 : cycle 600 부근에서 EOL 기준선에 도달하며 조기 종료 
    - 파란 계열 : cycle 1,000 이후에도 1.0Ah 수준을 유지하며 장수 

[to modeling]
- 초기 구간에는 모든 선이 거의 겹쳐 보임 -> QD 값만으로는 차이를 확인할 수 없음 -> ΔQ(V) 확인 필요 

### 3. 내부 저항(IR) 변화 - 열화 지표

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 사이클에 따른 IR 변화 (샘플 5개)
sample_cells = cell_ids[:10]
for cid in sample_cells:
    sub = df[df["cell_id"] == cid]
    axes[0].plot(sub["cycle"], sub["IR"], label=f"Cell {cid}", linewidth=1.2)

axes[0].set_xlabel("Cycle")
axes[0].set_ylabel("Internal Resistance IR (Ω)")
axes[0].set_title("IR - Cycle (Sampling#5)")
axes[0].legend(fontsize=9)

# cycle_life vs 초기 IR 상관관계
early_ir = df[df["cycle"] <= 10].groupby("cell_id")["IR"].mean().reset_index()
early_ir.columns = ["cell_id", "early_IR"]
merged = cycle_life_df.merge(early_ir, on="cell_id")

axes[1].scatter(
    merged["early_IR"], merged["cycle_life"], alpha=0.7, color="steelblue", edgecolors="white"
)
axes[1].set_xlabel("mean(IR) (Ω)")
axes[1].set_ylabel("Cycle Life")
axes[1].set_title("Early IR vs Cycle Life")

corr = merged["early_IR"].corr(merged["cycle_life"])
axes[1].text(
    0.05,
    0.95,
    f"Pearson r = {corr:.3f}",
    transform=axes[1].transAxes,
    fontsize=11,
    verticalalignment="top",
    color="tomato",
)

plt.suptitle("IR", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

[plot] - Sampling
- 내부저항(IR)이 전체 구간에서 평탄함. 배터리가 열화될수록 IR이 증가해야 하는데 해당 샘플에서는 확인되지 않음 
    - 샘플링 배터리는 IR 변화가 용량 변화에 비해 상대적으로 적음 -> 흑연 음극 구조 특징 
    - 사이클 800~900 부근의 스파이크는 측정 노이즈 또는 셀 교체 시점으로 해석 가능 

[plot] - correlation 
- 상관계수가 0.088이고, scatter 분포를 통해 상관관계 전혀 없음 

[to modeling]
- IR 단독으로 target variable 설명할 수 없음 
- IR 변화율(ΔIR) 또는 다른 변수와의 조합/비교 통한 파생변수 고민 

### 4. 충전 정책(C-rate)과 수명의 관계

In [ ]:
policy_life = cycle_life_df.groupby("charging_policy")["cycle_life"].agg(["mean", "std", "count"])
policy_life = policy_life.sort_values("mean", ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(policy_life))
bars = ax.bar(
    x,
    policy_life["mean"],
    yerr=policy_life["std"],
    color="steelblue",
    alpha=0.75,
    error_kw=dict(ecolor="gray", capsize=3),
)
ax.set_xticks(x)
ax.set_xticklabels(policy_life.index, rotation=45, ha="right", fontsize=9)
ax.set_xlabel("Charging Policy")
ax.set_ylabel("mean(Cycle Life)")
ax.set_title("average Cycle Life by C-rate (± 1 std)")

for bar, (_, row) in zip(bars, policy_life.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f"n={int(row['count'])}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.show()

[remark] C-rate : 배터리 용량 대비 총전/방전 속도를 나타내는 단위 
- C-rate 
    - `1C` : 1시간 만에 완충
    - `2C` : 30분 만에 완충 
    - `4C` : 15분 만에 완충 (숫자가 클수록 빠름) 
- `4C(80%)-4C`
    - `4C` : 1단계. 0% -> 80%까지 4C 속도로 충전 
    - `(80%)` : 전환 기준. 용량 80% 도달 시 속도 변경 
    - `4C` : 2단계. 나머지 20%는 4C 속도로 충전 
- 정책 비교 
    - `4C(80%)-4C` : 처음부터 끝까지 빠르게 
    - `4C(80%)-3.6C` : 2단계만 살짝 천천히 
    - `5.4C(80%)-5.4C` : 처음부터 끝까지 매우 빠르게 -> 수명 최단

[plot]
- 왼쪽(장수)에서 오른쪽(단명)으로 갈수록 C-rate 숫자가 커지는 경향 
    - 상위 구간 (cycle 1000~1200) : `4C(80%)-4C`, `3.6C(80%)-3.6C`
    - 중위 구간 (cycle 800~900)   : `6C(40%)-3C`, `5.4C(50%)-3.6C`
    - 하위 구간 (cycle 500~600)   : `8C(35%)-3.6C`, `5.4C(80%)-5.4C`
    - 낮은 C-rate은 천천히 충전할수록 수명이 길고, 높은 C-rate로 빠르게 충전할수록 수명이 짧음 
    - Notion > Domain : 빠른 충전이 음극 팽창/수축을 가속화한다는 내용 확인 
- 2단계 충전 전략 영향도 : `8C(15%)-3.6C` vs `8C(25%)-3.6C` 
    - 그래프에서는 전환 기준을 어떻게 가져가느냐에 따라 배터리 수명에 영향을 주는 것으로 보임 
    - 그러나 샘플이 정책당 2건 밖에 되지 않으므로 "경향성" 수준으로 해석해야 함 

[to modeling]
- 충전 정책은 배터리 수명에 영향을 미치고 있음 
- 특정 정책에서는 편차가 존재하므로 추가 변수 고려 필요

### 5. 상관관계 히트맵

In [ ]:
# 배터리별 초기 100 사이클 평균 집계
early = (
    df[df["cycle"] <= 100]
    .groupby("cell_id")
    .agg(
        cycle_life=("cycle_life", "first"),
        mean_QD=("QD", "mean"),
        std_QD=("QD", "std"),
        mean_IR=("IR", "mean"),
        mean_Tavg=("Tavg", "mean"),
        mean_Tmax=("Tmax", "mean"),
        mean_chargetime=("chargetime", "mean"),
    )
    .reset_index(drop=True)
)

corr_matrix = early.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr_matrix.iloc[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=color)

ax.set_title("Early 100 Cycle, Correlation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n[cycle_life Corr Rank]")
print(corr_matrix["cycle_life"].drop("cycle_life").sort_values(key=abs, ascending=False).round(3))

[correlation]
- `mean_chargetime` : 충전 시간이 길수록 수명이 길어짐 -> C-rate과 직결. 천천히 충전(긴 충전시간) = 낮은 C-rate = 긴 수명으로 앞에서 확인 
- `mean_Tavg`, `mean_Tmax` : 온도가 높을수록 수명이 짧아짐. 빠른 충전(높은 C-rate)이 발열을 유발하고 이것이 열화를 가속화하는 인과관계 확인 
- `mean_Tavg` vs `mean_Tmax` : 다중공선성(0.96) 

[to modeling]
- 선형 관계 일부분 확인 가능 
- 비선형 패턴까지 추가 확인 필요

---
## 5. 다음 단계 안내

이 Scratch 노트북에서 확인한 내용을 바탕으로 아래 작업을 수행하세요.

**DAY 1 - EDA**
1. `cycle_life` 분포는 어떻게 생겼는가?
2. 열화 곡선 - 방전 용량이 어떻게 감소하는가?
3. `ΔQ(V)` 곡선 - 초기 사이클에서 차이가 보이는가?
4. 충전 조건 (C-rate)과 수명의 관계는?
5. 상관관계 - 어떤 신호가 수명과 연관되어 있는가?

**DAY 2 - 모델링**
- Regression : `cycle_life` 수치 예측 (초기 100 사이클)
- Classification : 장/단 수명 이진 분류 (초기 5 사이클)

(**Hint**) : `ΔQ(V)` 곡선은 `cycles[n]['Qdlin']`을 사용하여 계산합니다.

# DAY 1 - 모델 전략 수립

- 대상 Dataset = Batch 1 + Batch 2 + Batch 3
- 아래 5개 질문에 대하여 각 Batch Set에 대한 EDA를 수행하고 Batch간 특징을 비교함
- EDA insight를 토대로 모델 설계 전략 수립

## Question

1. Cycle Life 분포는 어떻게 생겼는가?
    - 150 ~ 2,300 사이클 Histogram
    - 장수명(>1,000)/단수명(<500) 비율 확인
    - 이상치 셀 식별 - 왜 유독 짧은가?
2. 열화 곡선 - 방전 용량이 어떻게 감소하는가?
    - 사이클 별 Qd 추이 시각화
    - 열화 속도가 일정한가, 가속되는가?
    - Knee point - 급격한 열화 시작점 탐색
3. ΔQ(V) ****곡선 - 초기 사이클에서 차이가 보이는가?
    - 사이클 100번 - 사이클 10번의 Q(V) 차이 계산
    - 장수명 셀 vs 단수명 셀의 ΔQ 형태 비교
    - 이를 구분할 수 있는 통계값으로 피쳐 추출
4. 충전 조건 (C-rate)과 수명의 관계는?
    - 충전 프로토콜별 평균 수명 비교
    - 고속 충전 셀이 정말 수명이 짧은가?
    - 충전 전류 패턴과 열화 속도 상관 분석
5. 상관관계 - 어떤 신호가 수명과 연관되어 있는가?
    - 초기 사이클 피켜들과 Cycle Life 상관 계수 확인
    - 가장 강한 관계 식별
    - 멀티클리니어리티 문제 확인

### Memo

- Cell id 중복이 있는지 먼저 확인 해보았다.


In [ ]:
# normalize_batch / extract_summaries 구현은 skala_ds_mini_project.data에 있습니다.

In [ ]:
# 단일 배치 구조 확인용 대용량 참조를 해제하고, 각 파일을 순차 처리합니다.
# 여러 배치의 raw cycles를 동시에 보관하지 않습니다.
del batch, cell0, cycles_raw, cycles
batch_files = BATCH_FILES
CYCLE_EARLY, CYCLE_LATE = 10, 100
df1, dq_all_features, dq_curves, V = load_feature_inputs(
    DATA_DIR,
    batch_files,
    cycle_early=CYCLE_EARLY,
    cycle_late=CYCLE_LATE,
)

In [ ]:
# cell별 cycle life 하나만 사용
# cycle_life가 NaN인 셀(EOL 미도달)은 비율 계산에서 제외
cycle_life = df1.drop_duplicates(["cell_id"])["cycle_life"].dropna()

# 비율 계산
short_ratio = (cycle_life < 500).mean() * 100
long_ratio = (cycle_life > 1000).mean() * 100

# Histogram
plt.figure(figsize=(10, 5))

plt.hist(cycle_life, bins=15, edgecolor="white")

# 기준선
plt.axvline(500, linestyle="--", label=f"Short Life (<500): {short_ratio:.1f}%", color="tomato")

plt.axvline(1000, linestyle="--", label=f"Long Life (>1000): {long_ratio:.1f}%", color="cyan")

plt.title("Cycle Life Distribution")
plt.xlabel("Cycle Life")
plt.ylabel("Number of Battery Cells")

plt.legend()
plt.tight_layout()
plt.show()

print(f"Short Life (<500): {short_ratio:.1f}%")
print(f"Long Life (>1000): {long_ratio:.1f}%")

In [ ]:
import matplotlib.pyplot as plt

# cell별 cycle life 하나만 사용
df_cell = df1.drop_duplicates(["cell_id", "batch_id"]).copy()

# Life Group
df_cell["life_group"] = pd.cut(
    df_cell["cycle_life"],
    bins=[-float("inf"), 500, 1000, float("inf")],
    labels=["Short (<500)", "Normal (500~1000)", "Long (>1000)"],
)

# batch 목록
batch_ids = sorted(df_cell["batch_id"].unique())

# batch별 life group 비율
ratio = pd.crosstab(df_cell["batch_id"], df_cell["life_group"], normalize="index") * 100

# -------------------------------------------------
# Figure 생성
# -------------------------------------------------
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))

# =================================================
# 1. Histogram
# =================================================
for batch_id in batch_ids:
    data = df_cell.loc[df_cell["batch_id"] == batch_id, "cycle_life"]

    axes[0].hist(data, bins=15, alpha=0.5, label=f"Batch {batch_id}")

axes[0].axvline(500, linestyle="--")
axes[0].axvline(1000, linestyle="--")

axes[0].set_title("Cycle Life Distribution")
axes[0].set_xlabel("Cycle Life")
axes[0].set_ylabel("Number of Cells")
axes[0].legend()

# =================================================
# 2. Boxplot
# =================================================
box_data = [
    df_cell.loc[df_cell["batch_id"] == batch_id, "cycle_life"].dropna().values
    for batch_id in batch_ids
]

axes[1].boxplot(box_data, tick_labels=batch_ids)

axes[1].axhline(500, linestyle="--")
axes[1].axhline(1000, linestyle="--")

axes[1].set_title("Cycle Life by Batch")
axes[1].set_xlabel("Batch ID")
axes[1].set_ylabel("Cycle Life")

# =================================================
# 3. 100% Stacked Bar
# =================================================
bottom = [0] * len(ratio)

for group in ratio.columns:
    values = ratio[group].values

    axes[2].bar(ratio.index.astype(str), values, bottom=bottom, label=group)

    bottom = [b + v for b, v in zip(bottom, values)]

axes[2].set_title("Cycle Life Group Ratio by Batch")
axes[2].set_xlabel("Batch ID")
axes[2].set_ylabel("Ratio (%)")
axes[2].set_ylim(0, 100)
axes[2].legend(title="Life Group")

# -------------------------------------------------
# 전체 Figure
# -------------------------------------------------
fig.suptitle("Cycle Life Analysis by Batch", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Short-life 이상치 셀 식별 및 초기 특성 비교
# ==========================================

# cell 단위 정보
cell_info = (
    df1[["cell_id", "batch_id", "cycle_life", "charging_policy"]]
    .drop_duplicates(["cell_id", "batch_id"])
    .copy()
)

# 단수명 셀 정의
short_cells = cell_info[cell_info["cycle_life"] < 500].sort_values("cycle_life")

print("Short-life cells")
short_cells

In [ ]:
# ==========================================
# 단수명 셀 vs 나머지 셀
# ==========================================
short_ids = short_cells["cell_id"].tolist()

# 초기 100 cycle만 사용
early_df = df1[df1["cycle"] <= 100].copy()

# 그룹 구분
early_df["life_group"] = np.where(
    early_df["cell_id"].isin(short_ids), "Short Life (<500)", "Others"
)

features = ["QD", "QC", "IR", "Tmax", "Tavg", "Tmin", "chargetime"]

comparison = early_df.groupby("life_group")[features].mean().T

comparison = early_df.groupby("life_group")[features].mean().T

comparison["difference"] = comparison["Short Life (<500)"] - comparison["Others"]

comparison["ratio"] = comparison["Short Life (<500)"] / comparison["Others"]

comparison["pct_difference"] = comparison["difference"] / comparison["Others"] * 100

# 비율 차이가 큰 순서
comparison = comparison.reindex(
    comparison["pct_difference"].abs().sort_values(ascending=False).index
)

comparison.round(4)

In [ ]:
# 충전 정책 파싱은 공유 parse_policy 사용 (-newstructure 접미사 포함)
cell_info[["first_c", "switch_soc", "second_c"]] = cell_info["charging_policy"].apply(
    lambda x: pd.Series(parse_policy(x))
)

cell_info.groupby(["first_c", "switch_soc", "second_c"]).size()

In [ ]:
cell_info.groupby("switch_soc")["cycle_life"].agg(["count", "mean", "median"])

In [ ]:
cell_info.groupby("second_c")["cycle_life"].agg(["count", "mean", "median"])

### Cycle Life 분포는 어떻게 생겼는가?

#### 150 ~ 2,300 사이클 Histogram

- Batch 1~3 실제 범위는 **392 ~ 1,935 사이클** (n=129, 평균 834, 중앙값 828). 150 ~ 2,300은 전체 데이터셋 기준 범위로 보이며, 이번 대상 Dataset에는 150대 셀이 없음
- Batch별 분포 차이가 큼
    - Batch 1 : 중앙값 859 (534 ~ 1,227), 폭이 좁음
    - Batch 2 : 중앙값 472, 단수명에 집중
    - Batch 3 : 중앙값 1,006 (최대 1,935), 오른쪽 꼬리가 긺
- 전체 분포는 batch별 분포가 섞인 형태이고 right-skew → 모델링 시 `log10(cycle_life)` 타겟 사용

#### 장수명(>1,000)/단수명(<500) 비율 확인

~~Short Life (<500): 20.1%~~
~~Long Life (>1000): 25.9%~~

- (검증) 위 값은 `cycle_life`가 NaN인 10개 셀(Batch 2 : 8개, Batch 3 : 2개, EOL 미도달)이 분모에 포함되어 과소 계산됨 → 코드에서 NaN 제외하도록 수정
- NaN 제외(n=129) 기준
    - Short Life (<500) : **21.7%** (28개)
    - Long Life (>1000) : **27.9%** (36개)

#### 이상치 셀 식별 - 왜 유독 짧은가?

- 단수명 셀 28개가 **모두 Batch 2** → "단수명 vs 나머지" 비교가 사실상 "Batch 2 vs 나머지" 비교가 됨 (batch 교란)
- (검증) ~~초기의 100 사이클만 비교 해보았을 때, chargetime의 차이가 약 43% 정도로 나타나는 것을 알 수 있다.~~
    - 43%는 **평균** 기준이며, `chargetime`에는 극단 이상치(최대 3,934, 99.9% 분위 960, 중앙값 10.2)가 Batch 2에 몰려 있어 평균이 왜곡됨
    - Batch 2 내부에서 단수명 vs 나머지의 chargetime **중앙값은 10.18 vs 10.84** → 단수명 셀의 충전 시간이 길다는 근거가 없음
- "배치별 unique한 정책들이 존재하기 때문에 정책을 그대로 비교할 수 없다" → 타당. 여러 batch에 공통으로 있는 정책도 batch에 따라 수명이 크게 다름
    - `4.8C(80%)-4.8C` : Batch 1 753 vs Batch 2 484
    - `6C(60%)-3C` : Batch 1 744 vs Batch 2 400
    - → 정책 외에 Batch 2의 실험 조건 자체가 수명을 낮춤
- Batch 2 안에서는 평균 C-rate가 모두 약 4.8C로 같으므로, 총 충전 속도가 아니라 **전류 분배 패턴**의 차이로 봐야 함
    - 2단계 C-rate가 5C 이상인 정책(`3.6C(9%)-5C` 395, `3.6C(30%)-6C` 421, `4.65C(69%)-6C` 452, `4.65C(44%)-5C` 483)이 최하위
    - 단수명 셀 중 `-newstructure` 셀은 없음 (Batch 2의 `4.8C(80%)-4.8C-newstructure` 평균 872)
- → 모델링 시 batch 효과를 반드시 고려 (batch를 섞어서 train/test split, batch별 성능 확인)

In [ ]:
plt.figure(figsize=(12, 6))

for (batch_id, cell_id), group in df1.groupby(["batch_id", "cell_id"]):
    group = group.sort_values("cycle")

    plt.plot(group["cycle"], group["QD"], alpha=0.25)

plt.xlabel("Cycle")
plt.ylabel("Discharge Capacity (QD)")
plt.title("Discharge Capacity Degradation")
plt.tight_layout()
plt.show()

In [ ]:
# Outlier 필터링 후 열화 곡선

# 1. 초기 5 cycle 기준 공칭 용량 추정
nominal = estimate_nominal_capacity(df1)

lower = nominal * 0.33
upper = nominal * 1.33

print(f"공칭 용량(nominal) : {nominal:.4f} Ah")
print(f"필터 범위          : {lower:.4f} ~ {upper:.4f} Ah")

# 2. QD 이상치 필터링
df_clean = filter_capacity(df1, nominal=nominal)

removed = len(df1) - len(df_clean)

print(f"제거된 행 수       : {removed} ({removed / len(df1) * 100:.2f}%)")

# 3. 시각화
fig, ax = plt.subplots(figsize=(14, 6))

cells = df_clean[["batch_id", "cell_id"]].drop_duplicates()

colors = cm.coolwarm(np.linspace(0, 1, len(cells)))

for i, (_, cell) in enumerate(cells.iterrows()):
    batch_id = cell["batch_id"]
    cell_id = cell["cell_id"]

    sub = df_clean[
        (df_clean["batch_id"] == batch_id) & (df_clean["cell_id"] == cell_id)
    ].sort_values("cycle")

    ax.plot(sub["cycle"], sub["QD"], color=colors[i], linewidth=0.8, alpha=0.7)

# EOL 기준선
# EOL 기준선 : 데이터셋 cycle_life 정의 (1.1Ah의 80% = 0.88Ah)
ax.axhline(y=0.88, color="red", linestyle="--", linewidth=1.5, label="EOL (0.88 Ah)")

ax.set_xlabel("Cycle")
ax.set_ylabel("QD (Ah)")
ax.set_title("Battery Cell - QD Curve (Outlier Filtering)")

ax.set_ylim(0.7, 1.2)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
df_knee = filter_capacity(df1, lower=0.5, upper=1.2, nominal=nominal)

In [ ]:
# 전체 수명 기반 knee 진단: 공유 find_knee_point 사용. 모델 predictor에는 포함하지 않습니다.

In [ ]:
knee_points = (
    df_knee.groupby(["batch_id", "cell_id"]).apply(find_knee_point).reset_index(name="knee_cycle")
)

knee_points.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for (batch_id, cell_id), group in df_knee.groupby(["batch_id", "cell_id"]):
    group = group.sort_values("cycle")

    ax.plot(group["cycle"], group["QD"], linewidth=0.7, alpha=0.4)

    knee = find_knee_point(group)

    if not np.isnan(knee):
        knee_row = group.iloc[(group["cycle"] - knee).abs().argmin()]

        ax.scatter(knee_row["cycle"], knee_row["QD"], s=15)

ax.set_xlabel("Cycle")
ax.set_ylabel("QD (Ah)")
ax.set_title("QD Degradation and Knee Points")

plt.tight_layout()
plt.show()

### 2. 열화 곡선 - 방전 용량이 어떻게 감소하는가?

#### 사이클 별 Qd 추이 시각화

- 초기에는 1.07 ~ 1.10Ah 근처에서 거의 평탄하다가 후반부에 급락하는 형태
- 공칭 용량 1.077Ah 기준 0.33 ~ 1.33배를 벗어난 QD 이상치는 59행(0.05%)뿐
- (검증) EOL 기준선을 `nominal * 0.8 = 0.861Ah`로 그렸지만, 데이터셋의 `cycle_life`는 **QD = 0.88Ah**(1.1Ah의 80%) 도달 시점 기준임 (`cycle_life` 시점의 실제 QD 중앙값 0.881Ah) → 기준선을 0.88Ah로 수정

#### 열화 속도가 일정한가, 가속되는가?

급격하게 가속된다 → (검증) **타당**

- cycle 10 ~ EOL 구간을 3등분하여 구간별 QD 기울기 중앙값을 비교 : **-0.034 → -0.086 → -0.579 mAh/cycle** (마지막 구간이 첫 구간의 약 17배)
- 137개 셀 중 136개(99%)에서 마지막 구간이 첫 구간보다 가파름
- knee 이후 기울기가 knee 이전보다 중앙값 **9.7배** 큼
- → 비선형(가속) 열화이므로, 초기 구간 QD 기울기를 직선으로 외삽해서는 수명을 예측할 수 없음

#### Knee point - 급격한 열화 시작점 탐색

- 방법 : QD 곡선을 두 직선으로 나눴을 때 SSE가 최소가 되는 cycle (각 구간 최소 50 cycle)
- knee 중앙값 **637 cycle** (106 ~ 1,927), 2개 셀은 데이터 부족으로 검출 실패
- 수명 대비 위치 : 중앙값 **78%** (IQR 75 ~ 80%) → 수명의 약 3/4 지점에서 급격한 열화 시작
- knee vs cycle_life 상관 r = 0.96 → knee가 늦게 오는 셀이 오래 감
- 일부 셀(비율 최소 0.33)은 초기 노이즈로 knee가 너무 이르게 검출됨
- → knee는 초기 100 cycle 안에서 관측되지 않으므로 직접 피쳐로 쓸 수 없음. knee가 오기 훨씬 전에 수명을 예측할 수 있는 초기 신호(ΔQ(V))가 필요

In [ ]:
# load_feature_inputs에서 공유 함수로 계산한 ΔQ(V) 통계를 EDA에 사용
# batch_id/local_cell_id/cell_id 매핑과 알려지지 않은 target은 원본 테이블에 보존됩니다.
dq_features = dq_all_features.dropna(subset=["cycle_life"]).copy()
dq_features["life_group"] = pd.cut(
    dq_features["cycle_life"],
    bins=[-float("inf"), 500, 1000, float("inf")],
    labels=["Short (<500)", "Normal (500~1000)", "Long (>1000)"],
)
print(f"ΔQ(V) 계산된 셀 수 : {len(dq_features)}")
dq_features.head()

In [ ]:
# 장수명 셀 vs 단수명 셀의 ΔQ(V) 형태 비교

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. 전체 셀 ΔQ(V) - cycle life 색상
norm = plt.Normalize(dq_features["cycle_life"].min(), dq_features["cycle_life"].max())

for _, row in dq_features.iterrows():
    dq = dq_curves[(row["batch_id"], row["local_cell_id"])]

    axes[0].plot(dq, V, color=cm.coolwarm_r(norm(row["cycle_life"])), linewidth=0.8, alpha=0.7)

fig.colorbar(cm.ScalarMappable(norm=norm, cmap="coolwarm_r"), ax=axes[0], label="Cycle Life")
axes[0].set_xlabel(f"Q{CYCLE_LATE} - Q{CYCLE_EARLY} (Ah)")
axes[0].set_ylabel("Voltage (V)")
axes[0].set_title("ΔQ(V) Curve - All Cells")

# 2. 그룹별 평균 ΔQ(V) ± 1 std
for group, color in zip(["Short (<500)", "Long (>1000)"], ["tomato", "steelblue"]):
    sub = dq_features[dq_features["life_group"] == group]
    curves = np.vstack([dq_curves[(b, c)] for b, c in zip(sub["batch_id"], sub["local_cell_id"])])

    mean = curves.mean(axis=0)
    std = curves.std(axis=0)

    axes[1].plot(mean, V, color=color, label=f"{group} (n={len(sub)})")
    axes[1].fill_betweenx(V, mean - std, mean + std, color=color, alpha=0.2)

axes[1].set_xlabel(f"Q{CYCLE_LATE} - Q{CYCLE_EARLY} (Ah)")
axes[1].set_ylabel("Voltage (V)")
axes[1].set_title("ΔQ(V) Curve - Short vs Long (mean ± 1 std)")
axes[1].legend()

fig.suptitle("ΔQ(V) = Q100(V) - Q10(V)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ΔQ(V) 통계 피쳐 - 수명 구분력 확인

dq_cols = ["dQ_min", "dQ_mean", "dQ_var", "dQ_skew", "dQ_kurt", "dQ_2V"]

# log10 |ΔQ| 및 log10(target)은 공유 함수에서 계산 (0/비유한 값은 NaN)
log_cols = ["log_dQ_min", "log_dQ_mean", "log_dQ_var"]

# 1. 그룹별 평균 비교
print("[Life Group별 ΔQ 통계 평균]")
display(dq_features.groupby("life_group", observed=True)[dq_cols + log_cols].mean().T.round(4))

# 2. log10(cycle_life)와 상관계수
dq_corr = pd.DataFrame(
    {
        "corr": dq_features[dq_cols + log_cols].corrwith(dq_features["log_cycle_life"]),
    }
).sort_values("corr", key=abs, ascending=False)

print("[ΔQ 피쳐 vs log10(Cycle Life) 상관계수]")
display(dq_corr.round(3))

# 3. 가장 강한 피쳐 scatter (batch별 색상)
best = dq_corr.index[0]

fig, ax = plt.subplots(figsize=(8, 6))

for batch_id, sub in dq_features.groupby("batch_id"):
    ax.scatter(
        sub[best], sub["cycle_life"], alpha=0.7, edgecolors="white", label=f"Batch {batch_id}"
    )

ax.set_yscale("log")
ax.set_xlabel(best)
ax.set_ylabel("Cycle Life (log scale)")
ax.set_title(f"{best} vs Cycle Life (r = {dq_corr.loc[best, 'corr']:.3f})")
ax.legend()
plt.tight_layout()
plt.show()

### 3. ΔQ(V) ****곡선 - 초기 사이클에서 차이가 보이는가?

#### 사이클 100번 - 사이클 10번의 Q(V) 차이 계산

- `Qdlin`(3.6V → 2.0V, 1,000 point)으로 `ΔQ(V) = Q100(V) - Q10(V)` 계산 (cycle_life가 있는 129개 셀)
- 거의 모든 셀이 ΔQ < 0 (100 cycle 동안 용량 감소). 차이는 3.2V 부근에서 시작해 2.9 ~ 3.0V 부근에서 가장 크게 벌어짐
- Q2의 QD 곡선에서는 초기 100 cycle 구간이 거의 겹쳐 보였지만, ΔQ(V)에서는 **차이가 뚜렷하게 보임**

#### 장수명 셀 vs 단수명 셀의 ΔQ 형태 비교

- 곡선 **모양은 비슷하고 크기(진폭)가 다름**. 컬러맵에서 곡선이 수명 순서대로 정렬됨
- 단수명 vs 장수명 평균
    - `dQ_min` : -0.0615 vs -0.0202 Ah (약 3배)
    - `log10(var)` : -3.39 vs -4.37 (분산 약 10배)
    - 2.0V 끝값 : -0.016 vs -0.0034 Ah
- 평균 ± 1 std 밴드가 거의 겹치지 않음 → 100 cycle 시점에 이미 장/단수명 구분 가능

#### 이를 구분할 수 있는 통계값으로 피쳐 추출

- log10(cycle_life)와의 상관계수
    - `dQ_min` 0.888, `log_dQ_var` -0.886, `log_dQ_min` -0.874, `dQ_mean` 0.861
    - `dQ_skew`, `dQ_kurt`는 |r| < 0.33으로 약함
- batch별로도 `log_dQ_var` 상관 유지 : Batch 1 -0.87 / Batch 2 -0.92 / Batch 3 -0.76 → batch 차이에서 오는 가짜 상관이 아닌 실제 신호
- 대표 피쳐 : **`log10(var(ΔQ))`** (논문 결과와 일치), 보조 피쳐로 `dQ_min`

In [ ]:
# 충전 프로토콜별 평균 수명 비교
policy_life = (
    cell_info.dropna(subset=["cycle_life"])
    .groupby("charging_policy")
    .agg(
        mean=("cycle_life", "mean"),
        std=("cycle_life", "std"),
        count=("cycle_life", "size"),
        batches=("batch_id", lambda x: ",".join(map(str, sorted(x.unique())))),
    )
    .sort_values("mean")
)

fig, ax = plt.subplots(figsize=(10, 14))

y = range(len(policy_life))
ax.barh(
    y,
    policy_life["mean"],
    xerr=policy_life["std"].fillna(0),
    color="steelblue",
    alpha=0.75,
    error_kw=dict(ecolor="gray", capsize=3),
)
ax.set_yticks(y)
ax.set_yticklabels(
    [
        f"{p} [B{b}] (n={n})"
        for p, b, n in zip(policy_life.index, policy_life["batches"], policy_life["count"])
    ],
    fontsize=8,
)

ax.axvline(500, linestyle="--", color="tomato")
ax.axvline(1000, linestyle="--", color="cyan")

ax.set_xlabel("mean(Cycle Life)")
ax.set_title("Average Cycle Life by Charging Policy (± 1 std)")
plt.tight_layout()
plt.show()

policy_life.round(1)

In [ ]:
# 고속 충전 셀이 정말 수명이 짧은가? (공유 avg_c_rate)

cell_info["avg_c"] = avg_c_rate(
    cell_info["first_c"], cell_info["switch_soc"], cell_info["second_c"]
)

charge_df = cell_info.dropna(subset=["cycle_life", "avg_c"]).copy()

# Batch 2, 3은 모든 정책의 평균 C-rate가 약 4.8C (0 -> 80% 충전 시간 10분 고정)
# -> 평균 C-rate 차이가 존재하는 Batch 1에서만 고속/저속 비교
print(charge_df.groupby("batch_id")["avg_c"].agg(["min", "max", "std"]).round(3))

b1 = charge_df[charge_df["batch_id"] == 1].copy()

# Batch 1 평균 C-rate 3분위로 Low / Mid / High 구분
b1["c_group"] = pd.qcut(b1["avg_c"], q=3, labels=["Low", "Mid", "High"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. 평균 C-rate vs cycle life (전체 batch)
for batch_id, sub in charge_df.groupby("batch_id"):
    axes[0].scatter(
        sub["avg_c"], sub["cycle_life"], alpha=0.7, edgecolors="white", label=f"Batch {batch_id}"
    )

corr_b1 = b1["avg_c"].corr(b1["cycle_life"])

axes[0].set_xlabel("Average C-rate (0 -> 80% SOC)")
axes[0].set_ylabel("Cycle Life")
axes[0].set_title(f"Avg C-rate vs Cycle Life (Batch 1 : r = {corr_b1:.3f})")
axes[0].legend()

# 2. Batch 1 C-rate 그룹별 boxplot
groups = ["Low", "Mid", "High"]
axes[1].boxplot(
    [b1.loc[b1["c_group"] == g, "cycle_life"] for g in groups],
    tick_labels=[
        f"{g}\n({b1.loc[b1['c_group'] == g, 'avg_c'].min():.2f}"
        f"~{b1.loc[b1['c_group'] == g, 'avg_c'].max():.2f}C)"
        for g in groups
    ],
)
axes[1].set_ylabel("Cycle Life")
axes[1].set_title("Cycle Life by Avg C-rate Group (Batch 1)")

plt.tight_layout()
plt.show()

print(
    b1.groupby("c_group", observed=True)["cycle_life"]
    .agg(["count", "mean", "median", "std"])
    .round(1)
)

In [ ]:
# 충전 전류 패턴과 열화 속도 상관 분석 (공유 fade_slope)

# 열화 속도 지표
# - early_fade : 초기(10~100 cycle) QD 감소 기울기
# - knee_cycle : 급격한 열화 시작점 (Q2 결과 재사용)
degradation = pd.DataFrame(
    [
        {
            "batch_id": batch_id,
            "cell_id": cell_id,
            "early_fade": fade_slope(group),
            "early_chargetime": group.loc[
                group["cycle"] <= 100, "chargetime"
            ].median(),  # 극단 이상치 -> median
        }
        for (batch_id, cell_id), group in df_clean.groupby(["batch_id", "cell_id"])
    ]
)

degradation = degradation.merge(knee_points, on=["batch_id", "cell_id"], how="left").merge(
    charge_df[["batch_id", "cell_id", "cycle_life", "first_c", "switch_soc", "second_c", "avg_c"]],
    on=["batch_id", "cell_id"],
)

charge_cols = ["first_c", "switch_soc", "second_c", "avg_c", "early_chargetime"]
degrade_cols = ["early_fade", "knee_cycle", "cycle_life"]

# 충전 패턴(행) x 열화 지표(열) 상관계수
charge_degrade_corr = degradation[charge_cols + degrade_cols].corr().loc[charge_cols, degrade_cols]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(charge_degrade_corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(degrade_cols)))
ax.set_yticks(range(len(charge_cols)))
ax.set_xticklabels(degrade_cols, rotation=45, ha="right")
ax.set_yticklabels(charge_cols)

for i in range(len(charge_cols)):
    for j in range(len(degrade_cols)):
        val = charge_degrade_corr.iloc[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=color)

ax.set_title("Charging Pattern vs Degradation (Correlation)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

charge_degrade_corr.round(3)

### 4. 충전 조건 (C-rate)과 수명의 관계는?
#### 충전 프로토콜별 평균 수명 비교

- 하위 정책은 모두 Batch 2 : `3.6C(9%)-5C` 395, `3.6C(30%)-6C` 421, `5.6C(26%)-4.5C` 449, `4.65C(69%)-6C` 452
- 상위 : `4.8C(80%)-4.8C-newstructure` 1,333, `4C(80%)-4C` 1,227, `3.6C(80%)-3.6C` 1,182, Batch 3의 `-newstructure` 정책들
- 같은 정책도 batch에 따라 수명이 크게 다름
    - `4.8C(80%)-4.8C` : Batch 1 753 / Batch 2 484
    - `4.8C(80%)-4.8C-newstructure` : Batch 2 872 / Batch 3 1,564
    - → 정책 효과와 batch(실험 시기, 셀 구조) 효과가 섞여 있음
- 대부분 정책이 n = 2라서 "경향" 수준으로만 해석해야 함

#### 고속 충전 셀이 정말 수명이 짧은가?

- 0 → 80% 평균 C-rate를 계산하면 **Batch 2, 3은 모든 정책이 4.79 ~ 4.81C로 같음** (10분 충전 고정) → 고속/저속 비교는 Batch 1에서만 의미가 있음
- Batch 1 : Low(3.6 ~ 4.32C) 평균 987 → Mid 798 → High(4.75 ~ 5.4C) 722
    - 상관계수 -0.737, Low와 High의 평균 수명 차이 약 264 사이클 (중앙값 1,014 vs 720)
    - → **고속 충전 셀의 수명이 짧다는 것은 타당**
- 다만 평균 C-rate가 같은 Batch 2, 3에서도 수명이 392 ~ 1,935로 다양함 → 총 충전 속도 외에 전류 분배 패턴과 셀/실험 조건의 영향이 큼

#### 충전 전류 패턴과 열화 속도 상관 분석

- 전체 상관계수 : 정책 변수(`first_c`, `switch_soc`, `second_c`, `avg_c`, `early_chargetime`) vs `early_fade`/`knee_cycle`/`cycle_life` 모두 |r| ≤ 0.32 → 약함
- batch 안에서 `avg_c` vs `early_fade` : Batch 1 **-0.59** → C-rate가 높을수록 초기 용량 감소가 빠름
    - Batch 2, 3은 `avg_c`가 사실상 상수(std 0.005)라 batch 내 상관(-0.33, +0.27)은 해석할 수 없음
- 그러나 `early_fade`(10 ~ 100 cycle QD 기울기) 자체는 수명과 -0.13으로 약함 → 초기 QD 기울기보다 ΔQ(V)가 훨씬 강한 신호
- → 정책 파라미터는 보조 피쳐로만 사용하고, batch 효과를 함께 고려

In [ ]:
# 초기(2~100 cycle) summary + ΔQ + 정책 피쳐: 공유 24열 테이블
# labeled_only=False는 미관측 target/ΔQ를 NaN으로 남겨 추론 입력 구성에 사용합니다.
early_features = build_feature_table(df1, dq_all_features, labeled_only=True, nominal=nominal)
feature_cols = list(FEATURE_COLUMNS)  # 20 predictors, target/ID/knee는 제외
print(f"early_features shape : {early_features.shape}")
early_features.head()

In [ ]:
# 초기 사이클 피쳐 vs Cycle Life 상관 계수

life_corr = pd.DataFrame(
    {
        "corr(log_life)": early_features[feature_cols].corrwith(early_features["log_cycle_life"]),
    }
).sort_values("corr(log_life)", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(life_corr))
ax.barh(y, life_corr["corr(log_life)"], color="steelblue")

ax.set_yticks(y)
ax.set_yticklabels(life_corr.index)
ax.invert_yaxis()
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_xlim(-1, 1)
ax.set_xlabel("r (vs log10 Cycle Life)")
ax.set_title("Early-cycle Features vs Cycle Life", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

life_corr.round(3)

In [ ]:
# 가장 강한 관계 식별 - 상위 3개 피쳐 scatter

top_features = life_corr.index[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat in zip(axes, top_features):
    sub = early_features[[feat, "log_cycle_life", "batch_id"]].dropna()

    for batch_id, b in sub.groupby("batch_id"):
        ax.scatter(
            b[feat], b["log_cycle_life"], alpha=0.7, edgecolors="white", label=f"Batch {batch_id}"
        )

    # 단순 선형 회귀선 + R^2
    coef = np.polyfit(sub[feat], sub["log_cycle_life"], 1)
    xs = np.linspace(sub[feat].min(), sub[feat].max(), 100)
    r2 = sub[feat].corr(sub["log_cycle_life"]) ** 2

    ax.plot(xs, np.polyval(coef, xs), color="black", linestyle="--")
    ax.set_xlabel(feat)
    ax.set_ylabel("log10(Cycle Life)")
    ax.set_title(f"{feat} (R² = {r2:.3f})")
    ax.legend(fontsize=9)

fig.suptitle("Top 3 Features vs log10(Cycle Life)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 멀티클리니어리티 확인

X = early_features[feature_cols].dropna()
corr_matrix = X.corr()

# 1. 피쳐 간 상관 히트맵
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr_matrix.iloc[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

ax.set_title("Early-cycle Feature Correlation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 2. |r| > 0.8 인 피쳐 쌍
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
high_pairs = (
    upper_tri.stack()
    .rename("corr")
    .reset_index()
    .query("abs(corr) > 0.8")
    .sort_values("corr", key=abs, ascending=False)
)
print("[|r| > 0.8 피쳐 쌍]")
display(high_pairs.round(3))

# 3. VIF = 상관행렬 역행렬의 대각 원소 (VIF > 10 이면 다중공선성 의심)
vif = pd.Series(
    np.diag(np.linalg.pinv(corr_matrix.to_numpy())), index=corr_matrix.columns, name="VIF"
).sort_values(ascending=False)

print("[VIF]")
vif.round(2)

### 5. 상관관계 - 어떤 신호가 수명과 연관되어 있는가?

#### 초기 사이클 피켜들과 Cycle Life 상관 계수 확인

- 초기 100 cycle 요약값 + ΔQ 통계 + 충전 정책 변수, 총 20개 피쳐
- |상관계수(log10 수명)| 상위 : `log_dQ_var` -0.886, `log_dQ_min` -0.874, `log_dQ_mean` -0.820, `QD_init` -0.560, `QD_max_minus_init` -0.452
- 온도, IR, 정책 변수는 모두 |r| ≤ 0.32. `IR_mean`은 상관이 0으로 앞선 EDA 결과와 일치

#### 가장 강한 관계 식별

- **`log10(var(ΔQ))`** : R² ≈ 0.79, 단일 피쳐로 log 수명 분산의 약 79%를 설명. batch별로도 강한 상관 유지
- `QD_init`(-0.56)은 batch 내부 상관이 0.10 / -0.20 / 0.17로 약함 → Batch 3의 초기 용량이 낮고 수명이 긴 batch 간 차이에서 생긴 **가짜 상관** → 피쳐로 쓸 때 주의

#### 멀티클리니어리티 문제 확인

- |r| > 0.8인 쌍
    - `log_dQ_min` - `log_dQ_mean` 0.98, `log_dQ_min` - `log_dQ_var` 0.97, `log_dQ_mean` - `log_dQ_var` 0.93
    - `QD_diff_100_init` - `early_fade` 0.96
    - `chargetime_med` - `avg_c` -0.96
    - `Tavg_med` - `Tmax_med` 0.88, `Tavg_med` - `Tmin_med` 0.85
- VIF > 10 : `log_dQ_min` 372, `log_dQ_mean` 277, `log_dQ_var` 60, `Tavg_med` 54, `dQ_kurt` 25, `avg_c` 23, `Tmax_med` 21, `chargetime_med` 20, `early_fade` 19, `QD_diff_100_init` 18, `Tmin_med` 16, `dQ_skew` 10.5
- 대응
    - ΔQ 계열은 `log_dQ_var` 하나만 사용 (필요하면 `dQ_min` 추가)
    - 온도는 `Tavg` 제외, `chargetime_med`/`avg_c` 중 하나, `early_fade`/`QD_diff_100_init` 중 하나만 사용
    - 또는 Ridge / ElasticNet 같은 정규화 선형 모델로 공선성 흡수

#### to modeling

- Regression : 타겟 `log10(cycle_life)`, `log_dQ_var` 중심 + 소수 보조 피쳐, ElasticNet을 baseline으로 사용
- batch 효과가 크므로 batch를 섞어 split하고 batch별 오차를 따로 확인

In [ ]:
# ==========================================
# 모델 설계 근거 - 피쳐 선별 / Target 분포
# ==========================================
# 1. 피쳐별 상관계수 : 전체 vs batch 내부
#    -> batch마다 부호가 뒤집히는 피쳐는 batch 차이에서 생긴 상관일 가능성
batch_corr = pd.DataFrame(
    {"all": early_features[feature_cols].corrwith(early_features["log_cycle_life"])}
)

for batch_id, sub in early_features.groupby("batch_id"):
    batch_corr[f"batch{batch_id}"] = sub[feature_cols].corrwith(sub["log_cycle_life"])

batch_sign = np.sign(batch_corr[["batch1", "batch2", "batch3"]])
batch_corr["same_sign"] = batch_sign.nunique(axis=1) == 1

print("[피쳐 vs log10(Cycle Life) 상관계수 - 전체 / batch별]")
display(batch_corr.sort_values("all", key=abs, ascending=False).round(2))

# 2. 선택 피쳐 VIF 확인
selected = ["log_dQ_var", "dQ_kurt"]
candidates = ["chargetime_med", "QD_max_minus_init", "Tmin_med", "early_fade"]

sel_corr = early_features[selected + candidates].corr()
sel_vif = pd.Series(
    np.diag(np.linalg.pinv(sel_corr.to_numpy())), index=sel_corr.columns, name="VIF"
)
print("[선택 + 후보 피쳐 VIF]")
display(sel_vif.round(2))

# 3. Target 분포 : cycle_life vs log10(cycle_life)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(early_features["cycle_life"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title(f"cycle_life (skew = {early_features['cycle_life'].skew():.2f})")
axes[0].set_xlabel("Cycle Life")

axes[1].hist(early_features["log_cycle_life"], bins=20, color="steelblue", edgecolor="white")
axes[1].set_title(f"log10(cycle_life) (skew = {early_features['log_cycle_life'].skew():.2f})")
axes[1].set_xlabel("log10(Cycle Life)")

plt.suptitle("Target Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# DAY 1 - 모델 설계 전략

## 1. Feature Engineering

모든 피쳐는 **초기 100 cycle 이내 데이터만** 사용 (cycle 1은 측정 이상치가 많아 제외)

| 구분 | 피쳐 | 근거 |
| --- | --- | --- |
| **핵심** | `log_dQ_var` (log10 var(ΔQ(V))) | 상관계수 -0.89, batch별로도 -0.87 / -0.92 / -0.76으로 유지 (Q3, Q5) |
| **보조** | `dQ_kurt` | 상관 0.32로 약하지만 batch별 부호 일관 (0.27 / 0.65 / 0.34), `log_dQ_var`와 VIF 낮음 |
| **조건부 후보** | `chargetime_med`, `QD_max_minus_init`, `Tmin_med`, `early_fade` | batch마다 상관 부호가 뒤집힘 (예 : `chargetime_med` 0.73 / -0.93 / 0.60) → 모델 성능이 실제로 좋아질 때만 채택 |
| **제외** | `log_dQ_min`, `log_dQ_mean` | `log_dQ_var`와 r 0.93 ~ 0.98, VIF 277 ~ 372 (Q5) |
| | `QD_init` | 전체 상관 -0.56은 batch 간 차이에서 생긴 가짜 상관, batch 내부 0.10 / -0.20 / 0.17 (Q5) |
| | `Tavg_med`, `Tmax_med` | `Tmin_med`와 공선성 (r 0.85 ~ 0.88), 상관 약함 |
| | `QD_diff_100_init` | `early_fade`와 r 0.96 |
| | `IR_mean`, `IR_diff_100_init` | 수명과 상관 거의 0 (EDA 3, Q5) |
| | `first_c`, `switch_soc`, `second_c`, `avg_c` | 정책이 batch마다 다르고, Batch 2, 3은 `avg_c`가 상수 (Q4) → 새 batch에 일반화 어려움 |
| | `knee_cycle` | 수명의 약 78% 지점에서 관측 → 초기 100 cycle 안에서 알 수 없음 (Q2, leakage) |

## 2. Regression 선택 및 Target Variable

- 방법 : **Regression**
    - 수명을 사이클 수로 직접 예측하면 장/단수명 구분(classification)도 함께 할 수 있음
    - 핵심 피쳐 `log_dQ_var`와 수명의 관계가 선형에 가까움 (단일 피쳐 R² ≈ 0.79)
- Target : **`log10(cycle_life)`**
    - `cycle_life`는 right-skew (skew 0.90, Batch 3의 1,500 ~ 1,935 꼬리) → log 변환 후 skew -0.09
    - ΔQ 피쳐를 log 스케일로 쓸 때 수명과 선형 관계가 됨 (Q3)
    - 예측값은 `10 ** pred`로 되돌려 사이클 단위로 평가

## 3. Modeling Strategy

### 데이터 처리

- 대상 : Batch 1 ~ 3 중 `cycle_life`가 있는 **129개 셀** (EOL 미도달 10개 제외)
- QD 이상치 필터 (공칭 용량 0.33 ~ 1.33배), `chargetime`과 온도는 극단 이상치 때문에 median 사용
- 셀 수가 129개로 적음 → 피쳐 수를 2 ~ 6개로 제한하여 과적합 방지
- Train / Test 분할 : batch 비율을 유지하는 stratified split (예 : 70 / 30) + Train 내부 5-fold CV로 하이퍼파라미터 선택
    - 단수명 셀이 모두 Batch 2에 있으므로, batch를 섞지 않으면 Train/Test 분포가 크게 달라짐 (Q1)
- 일반화 확인 : Leave-One-Batch-Out (2개 batch로 학습, 나머지 1개 batch로 평가) → 새 실험 batch에서의 성능 하락 정도 확인
- 스케일링 : 선형 모델은 StandardScaler (Train에만 fit)

### 후보 모델

| 단계 | 모델 | 피쳐 | 목적 |
| --- | --- | --- | --- |
| Baseline 0 | 평균 예측 | - | 최소 기준 |
| Baseline 1 | Linear Regression | `log_dQ_var` | 단일 핵심 피쳐 성능 (논문의 variance model) |
| 주 모델 | Ridge | 핵심 + 보조 (+ 조건부 후보) | 셀 수가 적고 공선성이 있어 L1/L2 규제 필요, L1이 불필요한 후보 피쳐를 0으로 만들어 피쳐 선택 역할 |
| 비교 | Random Forest / Gradient Boosting | 동일 | 비선형 관계 확인, 다만 n = 129라 과적합 여부를 CV로 확인 |

- 피쳐 세트 비교 : `log_dQ_var` → + `dQ_kurt` → + 조건부 후보 순서로 추가하며 CV 오차가 줄어드는 경우만 채택

### 평가

- RMSE (cycle), MAPE (%) : 예측값을 사이클 단위로 되돌려 계산
- 잔차 분석 : batch별, 수명 구간별 (장수명 셀일수록 예측이 어려울 것으로 예상, EDA 1)
- 참고 : 현재 환경에는 scikit-learn이 없음 → DAY 2 시작 전 설치 필요